# Sección 5 – PatchTST e iTransformer con NeuralForecast (Nixtla)

Este notebook entrena y evalúa modelos basados en Transformers para forecasting de consumo eléctrico:

| Modelo | Descripción |
|--------|-------------|
| **PatchTST** | Transformer con parches sobre series de tiempo (Nie et al., 2023) |
| **iTransformer** | Transformer invertido (canal-wise attention) (Liu et al., 2024) |

**Protocolo:** Backtest rolling-origin con 92 orígenes (oct–dic 2014), horizonte H = 24 h.

## 0. Configuración e Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# NeuralForecast – modelos Transformer
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST, iTransformer

# Proyecto
from src.backtest import backtest
from src.metrics import evaluate_all, mae_per_horizon
from src.utils import set_seed

set_seed(42)
print('Imports OK')

## 1. Carga de Datos

In [ ]:
DATA_PATH = os.path.join(ROOT, 'data', 'processed')

panel_file = os.path.join(DATA_PATH, 'panel.parquet')
if not os.path.exists(panel_file):
    panel_file = os.path.join(DATA_PATH, 'panel.csv')

if os.path.exists(panel_file):
    if panel_file.endswith('.parquet'):
        panel = pd.read_parquet(panel_file)
    else:
        panel = pd.read_csv(panel_file, index_col=0, parse_dates=True)
    print(f'Panel cargado desde {panel_file}: {panel.shape}')
else:
    print('Panel no encontrado — descargando datos UCI...')
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00321/LD2011_2014.txt.zip'
    raw = pd.read_csv(url, sep=';', decimal=',', index_col=0, parse_dates=True)
    raw.index = pd.to_datetime(raw.index)
    raw = raw.resample('H').mean().dropna(axis=1, how='all')
    panel = raw.iloc[:, :20]
    os.makedirs(DATA_PATH, exist_ok=True)
    panel.to_parquet(os.path.join(DATA_PATH, 'panel.parquet'))
    print(f'Panel guardado: {panel.shape}')

if not isinstance(panel.index, pd.DatetimeIndex):
    panel.index = pd.to_datetime(panel.index)
panel = panel.sort_index()

print(f'Rango: {panel.index.min()} → {panel.index.max()}')
print(f'Clientes: {panel.shape[1]} | Pasos: {panel.shape[0]}')

In [ ]:
# ── Splits ────────────────────────────────────────────────────────────────────
TRAIN_END  = '2014-07-31 23:00:00'
VAL_END    = '2014-09-30 23:00:00'
TEST_START = '2014-10-01'
TEST_END   = '2014-12-31'
H          = 24
INPUT_SIZE = 168  # 7 días

panel_trainval = panel.loc[:VAL_END]
y_train_arr    = panel.loc[:TRAIN_END].values.T  # (N, T_train)

print(f'Train+Val: {panel_trainval.index.min()} → {panel_trainval.index.max()}')
print(f'Test     : {TEST_START} → {TEST_END}')

In [ ]:
# ── Conversión a formato Nixtla ───────────────────────────────────────────────
def panel_to_nixtla(df: pd.DataFrame) -> pd.DataFrame:
    long = df.reset_index().melt(
        id_vars=[df.index.name or 'index'],
        var_name='unique_id',
        value_name='y',
    ).rename(columns={df.index.name or 'index': 'ds'})
    long['ds'] = pd.to_datetime(long['ds'])
    long['y']  = long['y'].fillna(0.0)
    return long[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds']).reset_index(drop=True)

df_nixtla_trainval = panel_to_nixtla(panel_trainval)
print(f'Formato Nixtla train+val: {df_nixtla_trainval.shape}')
print(df_nixtla_trainval.head(3))

## 2. Definición de Modelos

### 2.1 PatchTST

**PatchTST** (Nie et al., 2023) divide la serie temporal en parches (*patches*) y aplica atención multi-cabeza sobre ellos:

- `patch_len=16` : longitud de cada parche (16 h)
- `stride=8`     : solapamiento 50 % entre parches
- `nhead=4`      : 4 cabezas de atención
- `hidden_size=64` : dimensión del modelo (equivalente a d_model)
- `n_layers=3`   : capas del Transformer encoder

In [ ]:
model_patchtst = PatchTST(
    h=H,
    input_size=INPUT_SIZE,
    # Parámetros de parcheo
    patch_len=16,
    stride=8,
    # Arquitectura Transformer
    n_heads=4,
    hidden_size=64,
    n_layers=3,
    # Regularización
    dropout=0.1,
    # Entrenamiento
    max_steps=500,
    learning_rate=1e-3,
    batch_size=32,
    random_seed=42,
    early_stop_patience_steps=10,
    val_check_steps=50,
    scaler_type='standard',
)

print(f'PatchTST definido: {model_patchtst}')

### 2.2 iTransformer

**iTransformer** (Liu et al., 2024) aplica atención **por canal** (variate-centric) en lugar de por tiempo, siendo especialmente eficaz en escenarios multivariados.

> **Nota:** cuando neuralforecast no incluye iTransformer, se usa TFT (Temporal Fusion Transformer) que es igualmente robusto.

In [ ]:
try:
    from neuralforecast.models import iTransformer as _iTransformer
    model_itransformer = _iTransformer(
        h=H,
        input_size=INPUT_SIZE,
        # Arquitectura
        n_heads=4,
        hidden_size=64,
        n_layers=3,
        dropout=0.1,
        # Entrenamiento
        max_steps=500,
        learning_rate=1e-3,
        batch_size=32,
        random_seed=42,
        early_stop_patience_steps=10,
        val_check_steps=50,
        scaler_type='standard',
    )
    TRANSFORMER2_NAME = 'iTransformer'
    print(f'iTransformer definido: {model_itransformer}')
except (ImportError, Exception) as e:
    print(f'iTransformer no disponible ({e}); usando TFT como alternativa.')
    from neuralforecast.models import TFT
    model_itransformer = TFT(
        h=H,
        input_size=INPUT_SIZE,
        # Arquitectura
        hidden_size=64,
        n_head=4,
        dropout=0.1,
        # Entrenamiento
        max_steps=500,
        learning_rate=1e-3,
        batch_size=32,
        random_seed=42,
        early_stop_patience_steps=10,
        val_check_steps=50,
        scaler_type='standard',
    )
    TRANSFORMER2_NAME = 'TFT'
    print(f'TFT definido: {model_itransformer}')

## 3. Entrenamiento (Train + Val)

In [ ]:
print('Entrenando PatchTST...')
nf_patchtst = NeuralForecast(models=[model_patchtst], freq='H')
nf_patchtst.fit(df=df_nixtla_trainval, val_size=H*30)
print('PatchTST entrenado ✓')

In [ ]:
print(f'Entrenando {TRANSFORMER2_NAME}...')
nf_itransformer = NeuralForecast(models=[model_itransformer], freq='H')
nf_itransformer.fit(df=df_nixtla_trainval, val_size=H*30)
print(f'{TRANSFORMER2_NAME} entrenado ✓')

## 4. Wrapper `NeuralForecastModel`

In [ ]:
class NeuralForecastModel:
    """
    Adapta NeuralForecast al protocolo ForecastModel de src/backtest.py.

    ForecastModel.predict(history, horizon, origin_time) → ndarray (N, H)
    """

    def __init__(self, nf: NeuralForecast, model_name: str,
                 clients: list, input_size: int = 168):
        self.nf         = nf
        self._name      = model_name
        self.clients    = list(clients)
        self.input_size = input_size

    @property
    def name(self) -> str:
        return self._name

    def predict(
        self,
        history: pd.DataFrame,
        horizon: int,
        origin_time: pd.Timestamp,
    ) -> np.ndarray:
        ctx = history.iloc[-self.input_size:]

        df_ctx = ctx.reset_index().melt(
            id_vars=[ctx.index.name or 'index'],
            var_name='unique_id',
            value_name='y',
        ).rename(columns={ctx.index.name or 'index': 'ds'})
        df_ctx['ds'] = pd.to_datetime(df_ctx['ds'])
        df_ctx['y']  = df_ctx['y'].fillna(0.0)
        df_ctx = df_ctx[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds'])

        preds_df = self.nf.predict(df=df_ctx)
        pred_col = [c for c in preds_df.columns if c not in ('unique_id', 'ds')][0]

        preds_pivot = (
            preds_df
            .sort_values(['unique_id', 'ds'])
            .pivot(index='unique_id', columns='ds', values=pred_col)
            .reindex(self.clients)
            .fillna(0.0)
        )
        return preds_pivot.values  # (N, H)


print('Wrapper NeuralForecastModel definido ✓')

## 5. Backtest Rolling-Origin en Prueba

In [ ]:
clients = list(panel.columns)

wrapped_patchtst     = NeuralForecastModel(nf_patchtst,     'PatchTST',         clients, INPUT_SIZE)
wrapped_itransformer = NeuralForecastModel(nf_itransformer, TRANSFORMER2_NAME,  clients, INPUT_SIZE)

print(f'Wrappers: {wrapped_patchtst.name}, {wrapped_itransformer.name} ✓')

In [ ]:
print('Ejecutando backtest PatchTST (92 orígenes)...')
result_patchtst = backtest(
    model=wrapped_patchtst,
    panel=panel,
    y_train=y_train_arr,
    H=H,
    test_start=TEST_START,
    test_end=TEST_END,
    seasonality=24,
    verbose=True,
    return_predictions=True,
)
print('Backtest PatchTST completado ✓')
print(pd.Series(result_patchtst['metrics']))

In [ ]:
print(f'Ejecutando backtest {TRANSFORMER2_NAME} (92 orígenes)...')
result_itransformer = backtest(
    model=wrapped_itransformer,
    panel=panel,
    y_train=y_train_arr,
    H=H,
    test_start=TEST_START,
    test_end=TEST_END,
    seasonality=24,
    verbose=True,
    return_predictions=True,
)
print(f'Backtest {TRANSFORMER2_NAME} completado ✓')
print(pd.Series(result_itransformer['metrics']))

## 6. Métricas de Prueba

In [ ]:
metrics_patchtst     = result_patchtst['metrics']
metrics_itransformer = result_itransformer['metrics']

metrics_df = pd.DataFrame([metrics_patchtst, metrics_itransformer])
display_cols = ['model', 'MAE', 'RMSE', 'WAPE', 'MASE', 'inference_time_s']
display_cols = [c for c in display_cols if c in metrics_df.columns]

print('\n📊 Métricas de prueba — Transformers:')
print(metrics_df[display_cols].to_string(index=False))

## 7. Guardar Predicciones

In [ ]:
OUTPUTS_DIR = os.path.join(ROOT, 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

preds_all = pd.concat([
    result_patchtst['predictions'],
    result_itransformer['predictions'],
], ignore_index=True)

out_path = os.path.join(OUTPUTS_DIR, 'predictions_transformers.csv')
preds_all.to_csv(out_path, index=False)
print(f'Predicciones guardadas: {out_path}')
print(f'Shape: {preds_all.shape}')
print(preds_all.head(3))

## 8. MAE por Paso del Horizonte (h = 1..24)

In [ ]:
mae_h_patchtst     = result_patchtst['mae_horizon']      # (H,)
mae_h_itransformer = result_itransformer['mae_horizon']  # (H,)

horizons = np.arange(1, H + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# — Panel izquierdo: curvas individuales ————————————————————————————————————
ax = axes[0]
ax.plot(horizons, mae_h_patchtst, marker='o', markersize=4, linewidth=2,
        color='mediumorchid', label='PatchTST')
ax.plot(horizons, mae_h_itransformer, marker='D', markersize=4, linewidth=2,
        color='darkorange',   label=TRANSFORMER2_NAME)
ax.set_xlabel('Paso del horizonte (h)', fontsize=12)
ax.set_ylabel('MAE (kWh)', fontsize=12)
ax.set_title('MAE por Horizonte — Transformers', fontsize=13, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

# — Panel derecho: diferencia (PatchTST − iTransformer) ——————————————————————
ax2 = axes[1]
delta = mae_h_patchtst - mae_h_itransformer
colors = ['tomato' if d > 0 else 'steelblue' for d in delta]
ax2.bar(horizons, delta, color=colors, edgecolor='gray', linewidth=0.5)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Paso del horizonte (h)', fontsize=12)
ax2.set_ylabel(f'MAE PatchTST − MAE {TRANSFORMER2_NAME} (kWh)', fontsize=11)
ax2.set_title(f'Diferencia de MAE\n(rojo = PatchTST peor; azul = {TRANSFORMER2_NAME} peor)',
              fontsize=12, fontweight='bold')
ax2.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(ROOT, 'outputs', 'figures', 'mae_horizon_transformers.png')
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {fig_path}')

## 9. Resumen Final

In [ ]:
print('=' * 65)
print('RESUMEN — Sección 5: PatchTST e iTransformer/TFT')
print('=' * 65)
for name, metrics in [('PatchTST', metrics_patchtst), (TRANSFORMER2_NAME, metrics_itransformer)]:
    print(f'\n{name}:')
    for k in ['MAE', 'RMSE', 'WAPE', 'MASE']:
        if k in metrics:
            print(f'  {k:10s}: {metrics[k]:.4f}')
print(f'\nPredicciones → outputs/predictions_transformers.csv')
print(f'Figura       → outputs/figures/mae_horizon_transformers.png')